In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.optimize import minimize
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm

import pymc as pm
import numpy as np
import aesara.tensor as at
%matplotlib inline

In [ ]:
def nfiPreprocessing(df, output_name):
    # 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
    code_species_dict = {11: ['소나무'], 12:['잣나무', '섬잣나무', '눈잣나무', '스트로브잣나무'], 13: ['일본잎갈나무', '잎갈나무'], 14: ['리기다소나무', '리기테다소나무', '방크스소나무'],
                        15: ['곰솔'], 16: ['전나무', '구상나무', '분비나무'], 17: ['편백', '화백'], 18: ['삼나무', '낙우송','메타세콰이아'], 19: ['가문비나무', '독일가문비나무', '종비나무'],
                        20: ['비자나무', '개비자나무'], 21: ['은행나무'], 31: ['상수리나무'], 32: ['신갈나무'], 33: ['굴참나무'], 34: ['갈찬나무', '떡갈나무', '졸참나무'],
                        35: ['오리나무', '물오리나무', '사방오리'], 36: ['고로쇠나무'], 37: ['자작나무', '거제수나무'],  38: ['박달나무', '개박달나무', '물박달나무'], 39: ['밤나무'],
                        40: ['물푸레나무', '들메나무', '물들메나무'], 41: ['서어나무', '개서어나무'], 42: ['때죽나무', '쪽동백나무'], 43: ['호두나무', '가래나무'], 44:['백합나무'], 
                        45: ['미루나무', '은사시나무', '이태리포플러나무', '수원사시나무'], 46: ['벚나무', '양벚나무', '산벚나무', '꽃벚나무', '왕벚나무', '잔털벚나무', '개벚나무', '올벚나무', '섬벚나무', '섬개벚나무', '산개벚지나무', '개벚지나무', ''], 47: ['느티나무'],  48:['층층나무', '곰의말채나무'],
                        49: ['아까시나무'], 61: ['가시나무', '붉가시나무', '종가시나무', '참가시나무', '개가시나무'], 62: ['구실잣밤나무'], 63: ['녹나무'], 64: ['굴거리나무'], 65: ['황칠나무'], 66: ['사스레피나무'], 67: ['후박나무'],
                         68: ['새덕이', '참식나무', '생달나무']}
    id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]
    name_lst1 = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', '포플러', '벚나무', 
                 '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
    name_lst2 = ['기타침엽수', '기타 참나무류', '기타활엽수']
    code_name_dict = {i: j for i, j in zip(id_lst, name_lst1)}

    # 임상도에 따라 NFI 수종명 재분류
    print(r"Reclassify based on Imsang code...")
    nfi_names = df['수종명'].unique()
    nfi_imsang = [df.loc[(df['수종명']==name), '침활구분'].unique()[0] for name in nfi_names]
    nfi_dict = {i : j for i, j in zip(nfi_names, nfi_imsang)}

    # 속성 추출 및 단위 환산
    print("Extract necessary columns & Convert Unit...")
    df2 = df[['표본점번호', '조사차기', '수종명', '침활구분', '흉고직경', '수고', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']]
    cm_to_inch = 0.3937
    cm_to_ft = 0.0328084
    df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
    df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
    df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)
    df2['해발고(m)'] = df2['해발고(m)'] / 100 # hm로 변환
    df2['경사(degree)'] = np.tan(np.radians(df2['경사(degree)'])) # tangent로 변환
    df2['방위각(º)'] = np.radians(df2['방위각(º)']) # radian으로 변환
    df2['평균수관밀도(%)'] = df2['평균수관밀도(%)'] / 100 # 소수점 자릿수로 변환
    # ['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']
    df2.columns = ['SampleID', 'Cycle', 'Species', 'Imsang', 'DBH(inch)', 'H(ft)', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long']
    df2.info

    # 수관높이비율(Crown ratio) 생성
    print("Add new columns: Crown Ratio, Crown Height, Imsang code, Imsang species name...")
    df2.insert(6, 'CR', (df2['CBH(ft)'] / df2['H(ft)']))
    df2.insert(7, 'CH', (df2['H(ft)'] - df2['CBH(ft)']))
    # 임상도 기준 수종명 및 수종코드 칼럼 삽입
    df2['I_Species'] = np.full(len(df2), '-99')
    df2['SID'] = np.full(len(df2), -99)
    for key, name_lst in code_species_dict.items(): 
        condition = df2['Species'].isin(name_lst)
        df2.loc[condition, 'I_Species'] = code_name_dict[key]
        df2.loc[condition, 'SID'] = key
        
        condition2 = ((df2['Imsang'] == '활엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition2, 'I_Species'] = '기타활엽수'
        df2.loc[condition2, 'SID'] = 30
    
        condition3 = ((df2['Imsang'] == '침엽수') & (df2['I_Species'] == '-99'))
        df2.loc[condition3, 'I_Species'] = '기타침엽수'
        df2.loc[condition3, 'SID'] = 10

    print(f"Save the dataframe at {output_name}")
    df_fin = pd.concat([df2[df2.columns[-2:]], df2[df2.columns[:3]], df2[df2.columns[3:15]]], axis=1)
    print(df_fin.shape)
    df_fin = df_fin.dropna()
    df_fin.to_csv(os.path.join(output_name), encoding='cp949', index=False)

    return df_fin

In [ ]:
df_clean = nfiPreprocessing(df, r"D:/ForestFire/CBH/data/NFI6-7_cleaned.csv")

In [ ]:
df_clean.info()

In [ ]:
df_clean.head(3)

# Build Hierarchical Model

In [ ]:
# 함수 정의
# funciton with log transformation of DBH & Height
def func2(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log/D_log)+(b2 * H_log)+(b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

# Assuming you have numpy arrays: H, D, EL, SL, AZ, CD, species_idx, y

with pm.Model() as model:
    # Hyperpriors
    mu_a = pm.Normal("mu_a", mu=0, sigma=1)
    sigma_a = pm.HalfNormal("sigma_a", sigma=1)
    mu_b1 = pm.Normal("mu_b1", mu=0, sigma=1)
    sigma_b1 = pm.HalfNormal("sigma_b1", sigma=1)

    # Species-level parameters
    a = pm.Normal("a", mu=mu_a, sigma=sigma_a, shape=n_species)
    b1 = pm.Normal("b1", mu=mu_b1, sigma=sigma_b1, shape=n_species)

    # Global parameters
    b2 = pm.Normal("b2", mu=0, sigma=1)
    b3 = pm.Normal("b3", mu=0, sigma=1)
    c1 = pm.Normal("c1", mu=0, sigma=1)
    d1 = pm.Normal("d1", mu=0, sigma=1)

    H_log, D_log = np.log1p(H), np.log1p(D)
    size = b1[species_idx] * (H_log / D_log) + b2 * H_log + b3 * D_log**2
    comp = c1 * CD
    site = d1 * EL

    x = a[species_idx] + size + comp + site
    cr = 1 / (1 + at.exp(-x))

    pm.Normal("obs", mu=cr, sigma=0.05, observed=y)

    trace = pm.sample(draws=1000, tune=1000, chains=2, target_accept=0.9)


In [ ]:
# read training dataset
data_dir = r"D:\ForestFire\CBH\data"
file_name = r"NFI6-7_cleaned.csv"
df_all = pd.read_csv(os.path.join(data_dir, file_name), encoding='cp949')
df_all = df_all.reset_index()

In [ ]:
# Map names to codes
name_lst1 =  ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', 
             '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', 
             '포플러', '벚나무', '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스레피나무', '후박나무','새덕이']
name_lst2 = ['기타침엽수', '기타참나무류', '기타활엽수']  # same as your input
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]     # same as your input
code_name_dict = {j: i for i, j in zip(id_lst, name_lst1)}
code_name_dict.update({'기타침엽수': 10, '기타활엽수': 30, '기타참나무류': 34})
reversed_dict = {value : key for key, value in code_name_dict.items()}

In [ ]:
# Configuration
try_num = "Try1_lam0.2"
lam = 0
test_ratio = 0.1 # 0.15
n_fold = 5
result_dir = r'D:/ForestFire/CBH/result/Baseline2'
os.makedirs(result_dir, exist_ok=True)


target_trees = np.unique(df_all.SID) # [code_name_dict[i] for i in valid_species]
rec = np.zeros((len(target_trees), 21), dtype=object)
rec[:, 0] = target_trees

# Collector for unified test set
total_test_list = []
total_train_list = []

for i, sid in enumerate(tqdm(target_trees), 1):
    print(f"[{i}/{len(target_trees)}] Processing SID: {sid}")
    condition = (df_all['SID'] == sid)
    nfi6 = df_all.query("Cycle == 6").loc[condition]
    nfi7 = df_all.query("Cycle == 7").loc[condition]

    cnt = (len(nfi6), len(nfi7))
    print(cnt)
    if nfi6.isnull().values.any() or nfi7.isnull().values.any():
        print("Null data detected, skipping SID:", sid)
        continue

    # Split train/test
    if len(nfi6) >= 180:
        nfi6_test = nfi6.sample(frac=test_ratio)
        nfi7_test = nfi7.sample(frac=test_ratio)
        df_train = pd.concat([nfi6.drop(nfi6_test.index).assign(Cycle=6),nfi7.drop(nfi7_test.index).assign(Cycle=7)])

    elif (len(nfi6) < 180) & (len(nfi7) >= 30):
        nfi6_test = nfi6
        nfi7_test = nfi7.sample(frac=test_ratio)
        df_train = nfi7.drop(nfi7_test.index).assign(Cycle=7)

    else:
        print(f"Skip {sid}: Not enough number of the samples")
        best_params = [np.nan] * 4
        rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan
        ] + list(best_params)
        continue
        
    # test-train dataset list에 저장    
    total_test_list.extend([nfi6_test.assign(SID=sid, Cycle=6), nfi7_test.assign(SID=sid, Cycle=7)])
    total_train_list.extend([df_train.assign(SID=sid, Cycle=7)])

    # X, y array 만들기
    cols = ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'CR', 'Cycle']
    X = df_train[cols].drop(columns=['CR'])
    y = df_train['CR']
    stratify_col = df_train['Cycle']

    # score 변수 initialization
    best_score, worst_score, best_params = -np.inf, np.inf, None
    cv_scores = []

    print("lenght of training dataset: ", len(X))
    if len(X) >= 30:
        kf = StratifiedKFold(n_splits=n_fold, shuffle=True) # , random_state=100
        for train_idx, val_idx in kf.split(X, stratify_col):
            X_train = X.iloc[train_idx].drop(columns='Cycle').to_numpy().T
            y_train = y.iloc[train_idx]
            popt, _ = curve_fit(func3, X_train, y_train)
            result_reg = minimize(loss_func3, x0=popt, args=(lam, X_train, y_train))
            opt_params = result_reg.x
            score = r2_score(y_train, func3(X_train, *opt_params))
            cv_scores.append(score)
            if score > best_score:
                best_score = score
                best_params = opt_params
            if score < worst_score:
                worst_score = score
    else: continue

    # Evaluate on full training data
    X_full = X.drop(columns='Cycle').to_numpy().T
    y_full = y
    y_pred_full = func3(X_full, *best_params)
    r2_train = r2_score(y_full, y_pred_full)
    mae_train = mean_absolute_error(y_full, y_pred_full)
    rmse_train = root_mean_squared_error(y_full, y_pred_full)

    # Evaluate on test sets
    X6_test = nfi6_test[cols].drop(columns=['CR', 'Cycle']).to_numpy().T
    y6_test = nfi6_test['CR']
    X7_test = nfi7_test[cols].drop(columns=['CR', 'Cycle']).to_numpy().T
    y7_test = nfi7_test['CR']

    # test dataset 예측 및 score 산출 (NFI6, NFI7, NFI6+7)
    y_pred6 = func3(X6_test, *best_params)
    y_pred7 = func3(X7_test, *best_params)
    r2_test_nfi6 = r2_score(y6_test, y_pred6)
    r2_test_nfi7 = r2_score(y7_test, y_pred7)
    mae_test_nfi6 = mean_absolute_error(y6_test, y_pred6)
    mae_test_nfi7 = mean_absolute_error(y7_test, y_pred7)
    rmse_test_nfi6 = root_mean_squared_error(y6_test, y_pred6)
    rmse_test_nfi7 = root_mean_squared_error(y7_test, y_pred7)
    y_all_true = np.concatenate([y6_test, y7_test])
    y_all_pred = np.concatenate([y_pred6, y_pred7])
    r2_test_all = r2_score(y_all_true, y_all_pred)
    mae_test_all = mean_absolute_error(y_all_true, y_all_pred)
    rmse_test_all = root_mean_squared_error(y_all_true, y_all_pred)

    # record list에 score 결과 저장
    rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.mean(cv_scores), best_score, worst_score,
        r2_train, mae_train, rmse_train,
        r2_test_all, mae_test_all, rmse_test_all,
        r2_test_nfi6, mae_test_nfi6, rmse_test_nfi6,
        r2_test_nfi7, mae_test_nfi7, rmse_test_nfi7
    ] + list(best_params)

# Save results into a textfile
header = ['SID', 'Count', 'CV_Mean', 'CV_Best', 'CV_Worst',
        'R2_Train', 'MAE_Train', 'RMSE_Train',
        'R2_Test_All', 'MAE_Test_All', 'RMSE_Test_All',
        'R2_Test_NFI6', 'MAE_Test_NFI6', 'RMSE_Test_NFI6',
        'R2_Test_NFI7', 'MAE_Test_NFI7', 'RMSE_Test_NFI7']
np.savetxt(
    os.path.join(result_dir, f'CR_Model_Result_Try{try_num}.txt'),
    rec, delimiter=',', fmt='%s',
    header=','.join(header + [f'Coef{i+1}' for i in range(11)]),
    comments=''
)

# Save unified test dataset
if total_test_list:
    pd.concat(total_test_list).to_csv(os.path.join(result_dir, f"NFI6+7_test_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("⚠️ Warning: No test data collected, skipping test dataset save.")

# Save unified train dataset
if total_train_list:
    pd.concat(total_train_list).to_csv(os.path.join(result_dir, f"NFI6+7_train_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("⚠️ Warning: No test data collected, skipping test dataset save.")


In [ ]:
df_result = pd.DataFrame(rec, columns=header + [f'Par{i}' for i in range(4)])
s_names = [reversed_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', s_names)
r2_columns = ['CV_Mean', 'CV_Best', 'CV_Worst', 'R2_Train', 'R2_Test_All', 'R2_Test_NFI6', 'R2_Test_NFI7']
df_result.loc[:, r2_columns] = df_result.loc[:, r2_columns].map(lambda x: 0 if x < 0 else x) # df_result.iloc[:, 4:19] = df_result.iloc[:, :19].clip(lower=0)
df_result.to_csv(os.path.join(result_dir, f'CR_Model_Result_{try_num}.csv'), encoding='cp949')

In [ ]:
sns.pairplot(df_all[cols].drop(columns=['Cycle']))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# Set font for Korean
plt.rc('font', family='Malgun Gothic')  
plt.rcParams['axes.unicode_minus'] = False  

# Read data
title = f"CR_Model_Result_Try{try_num}"
df_result = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')

# Set figure
plt.figure(figsize=(20, 6))
bar_width = 0.15
x = np.arange(len(df_result))

# Plot bars side-by-side
plt.bar(x - 2*bar_width, df_result["CV_Mean"], width=bar_width, label="CV_Mean", color="blue", alpha=0.7)
plt.bar(x - bar_width, df_result["R2_Train"], width=bar_width, label="R2_Train", color="orange", alpha=0.7)
plt.bar(x, df_result["R2_Test_All"], width=bar_width, label="R2_Test_All", color="purple", alpha=0.7)
plt.bar(x + bar_width, df_result["R2_Test_NFI6"], width=bar_width, label="R2_Test_NFI6", color="pink", alpha=0.7)
plt.bar(x + 2*bar_width, df_result["R2_Test_NFI7"], width=bar_width, label="R2_Test_NFI7", color="skyblue", alpha=0.7)

# Annotate values above bars
for i in range(len(df_result)):
    plt.text(x[i] - 2*bar_width, df_result["CV_Mean"][i] + 0.01, f'{df_result["CV_Mean"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] - bar_width, df_result["R2_Train"][i] + 0.01, f'{df_result["R2_Train"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i], df_result["R2_Test_All"][i] + 0.01, f'{df_result["R2_Test_All"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + bar_width, df_result["R2_Test_NFI6"][i] + 0.01, f'{df_result["R2_Test_NFI6"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + 2*bar_width, df_result["R2_Test_NFI7"][i] + 0.01, f'{df_result["R2_Test_NFI7"][i]:.2f}', ha='center', fontsize=6)

# X labels and layout
x_labels = [f"{df_result.loc[i, 'SName']}{df_result.loc[i, 'Count']}" for i in range(len(df_result))]
plt.xticks(ticks=x, labels=x_labels, rotation=45, ha="right")
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.legend()
plt.tight_layout()

# Save and show
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()


# Compare Azimuth Distribution: 그냥 NFI 값 쓰는 걸로!

In [ ]:
import geopandas as gpd

In [ ]:
gpd_nfi = gpd.read_file(r"D:\ForestFire\CBH\data\NFI67_Aspect_5181.shp")
print(gpd_nfi.info())
gpd_nfi_clean = gpd_nfi.query("RASTERVALU != -1 & RASTERVALU != 0")

In [ ]:
gpd_clean = gpd_nfi_clean.merge(df, on = "표본점번호",how="inner")
gpd_clean.

In [ ]:
az_true = gpd_clean["Azimuth(rad)"]
az_pred = gpd_clean['rasterval(rad)']
mean_val = (az_true + az_pred) / 2
deviation = az_pred - az_true
mean_diff = np.mean(mean_val)
std_dev = np.std(deviation, ddof=1)
upper_limit = mean_diff + 1.96 * std_dev
lower_limit = mean_diff - 1.96 * std_dev

plt.figure(figsize = (8, 6))
sns.scatterplot(x=mean_val, y= deviation, alpha=0.7)
plt.axhline(mean_diff, color="gray", linestyle="--", label=f"Mean = {mean_diff:.2f}")
plt.axhline(upper_limit, color="blue", linestyle="--", label=f"+1.96 SD = {upper_limit:.2f}")
plt.axhline(lower_limit, color="red", linestyle="--", label = f"-1.96 SD = {lower_limit:.2f}")
plt.xlabel("Mean of Observed and Terrain-driven")
plt.ylabel("Difference between Terrain-driven and ")
plt.title("Bland-Altman Plot")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(x=az_true, y=az_pred, alpha=0.7)

In [ ]:
gpd_clean